# 6장 1강: CV 기반 하이퍼파라미터 튜닝과 모델 탐색

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 하이퍼파라미터 탐색과 교차검증을 결합해 최적 모델을 찾습니다.

- GridSearchCV와 RandomizedSearchCV의 탐색 방식과 비용 차이를 비교합니다.
- Optuna의 TPE 탐색을 이용해 초기 탐색 이후 이전 trial 결과를 활용하는 탐색을 수행합니다.
- 교차검증 평균 점수와 표준편차를 함께 확인합니다.
- 탐색 시간까지 고려하여 최종 후보를 선정합니다.
- 처음부터 분리해 둔 hold-out test set은 최종 모델을 정한 뒤 한 번만 사용합니다.
- 최종 모델과 메타데이터를 저장합니다.

> 이 실습에서는 6장 1강 교안에서 다룬 Grid Search, Random Search, Optuna, CV 평균·표준편차, 탐색 비용, hold-out test, 모델 저장 내용만 사용합니다.


## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- scipy
- Optuna
- joblib
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

하이퍼파라미터 탐색에 Test 데이터를 사용하면 안 됩니다.

따라서 먼저 전체 데이터를 **Train / hold-out Test**로 분리하고,

- 하이퍼파라미터 탐색과 교차검증 → Train 데이터만 사용
- 최종 성능 확인 → 모든 선택이 끝난 뒤 hold-out Test를 한 번만 사용

하는 흐름으로 진행합니다.

In [6]:
import os
import json
import time
import joblib
import optuna
import pandas as pd

from datetime import datetime
from scipy.stats import randint

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Hold-out Test 크기:", X_test.shape)

Train 크기: (1168, 12)
Hold-out Test 크기: (292, 12)


In [2]:
pip install optuna

  Using cached colorlog-6.12.0-py3-none-any.whl.metadata (11 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.4 MB 3.7 MB/s eta 0:00:01
   --------------------- ------------------ 1.3/2.4 MB 3.6 MB/s eta 0:00:01
   ---------------------------------- ----- 2.1/2.4 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 3.5 MB/s  0:00:00
Using cached colorlog-6.12.0-py3-none-any.whl (12 kB)

   ---------------------------------------- 0/5 [sqlalchemy]
   ---------------------------------------- 0/5 [sqlalchemy]
   ---------------------------------------- 0/5 [sqlalchemy]
   ---------------------------------------- 0/5 [sqlalchemy]
   ---------------------------------------- 0/5 [sqlalchemy]
   ---------------------------------------- 0/5 [sqlalchemy]
   ---------------------------------------- 0/5 [sqlalchemy]
   ---------------------------------------- 0/5 [sqlalchemy]
   -------------------

---

# 필수 1. Grid Search와 Random Search 비교

## 배경

GridSearchCV는 지정한 모든 조합을 빠짐없이 평가하고, RandomizedSearchCV는 정해진 횟수만큼 조합을 무작위로 선택합니다.

이번 실습에서는 같은 RandomForestRegressor를 대상으로 두 탐색 전략을 비교합니다.

## 문제 1. GridSearchCV를 수행하세요.

### 요구사항

1. `RandomForestRegressor(random_state=42, n_jobs=-1)`를 사용합니다.
2. 다음 후보를 사용합니다.

```python
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}
```

3. `cv=5`를 사용합니다.
4. `scoring="r2"`를 사용합니다.
5. 탐색 시간을 측정합니다.
6. `best_params_`와 `best_score_`를 출력합니다.
7. `cv_results_`에서 최적 조합의 `mean_test_score`, `std_test_score`를 확인합니다.

### 확인 포인트

- 총 9개 조합을 모두 평가하는가?
- 5-Fold이므로 각 조합이 5번 평가되는 구조를 이해했는가?
- 평균 점수뿐 아니라 표준편차도 확인했는가?

In [7]:
# TODO
# GridSearchCV를 수행하세요.

grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=grid_params,
    cv=5,
    scoring="r2"
)

start = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - start

best_idx = grid_search.best_index_

print(f"평가한 조합 수  : {len(grid_search.cv_results_['params'])}개")
print(f"총 학습 횟수    : {len(grid_search.cv_results_['params']) * 5}회 (조합 수 x cv 5)")
print(f"탐색 시간       : {grid_time:.2f}초")
print(f"best_params_    : {grid_search.best_params_}")
print(f"best_score_     : {grid_search.best_score_:.4f}")
print(f"mean_test_score : {grid_search.cv_results_['mean_test_score'][best_idx]:.4f}")
print(f"std_test_score  : {grid_search.cv_results_['std_test_score'][best_idx]:.4f}")

평가한 조합 수  : 9개
총 학습 횟수    : 45회 (조합 수 x cv 5)
탐색 시간       : 7.21초
best_params_    : {'max_depth': 12, 'n_estimators': 200}
best_score_     : 0.8206
mean_test_score : 0.8206
std_test_score  : 0.0644


최고 CV 평균(best_scotre)  
0.82, max_depth(12)와 n_estiamtors(200)의 조합으로 5개의 검증 R^2을 평균한 값  

CV 표준편차(std_test_score)  
0.06, 최적 조합 fold별로 검증 점수가 평균 주변에서 얼마나 달랐는지를 나타냄  

### Q1. Grid Search의 가장 큰 장점과 한계는 무엇인가요?

**답변:**  
- 장점 : 지정한 조합을 빠짐없이 평가, 하이퍼 파라미터 튜닝 최적화를 용이하게 찾음  
- 담점 : 후보 수가 많아지면 탐색비용이 매우 느려짐  

## 문제 2. RandomizedSearchCV를 수행하세요.

### 요구사항

1. 같은 RandomForestRegressor를 사용합니다.
2. 다음 탐색 공간을 사용합니다.

```python
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}
```

3. `n_iter=9`를 사용합니다.
4. `cv=5`, `scoring="r2"`를 사용합니다.
5. `random_state=42`를 사용합니다.
6. 탐색 시간을 측정합니다.
7. 최적 파라미터, CV 평균 점수, 표준편차를 확인합니다.

### 확인 포인트

- 전체 조합을 모두 보는 것이 아니라 9번만 무작위로 탐색하는가?
- `n_iter`가 탐색 예산을 직접 정한다는 점을 이해했는가?

In [9]:
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=random_params,
    n_iter=9,
    cv=5,
    scoring="r2",
    random_state=42
)

start = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - start

best_idx = random_search.best_index_
n_space = (301 - 100) * (13 - 4)

print(f"탐색 공간 전체 조합 수 : {n_space:,}개")
print(f"실제 평가한 조합 수    : {random_search.n_iter}개 (n_iter)")
print(f"탐색 비율              : {random_search.n_iter / n_space * 100:.3f}%")
print(f"총 학습 횟수           : {random_search.n_iter * 5}회 (n_iter x cv 5)")
print(f"탐색 시간              : {random_time:.2f}초")
print(f"best_params_           : {random_search.best_params_}")
print(f"best_score_            : {random_search.best_score_:.4f}")
print(f"mean_test_score        : {random_search.cv_results_['mean_test_score'][best_idx]:.4f}")
print(f"std_test_score         : {random_search.cv_results_['std_test_score'][best_idx]:.4f}")

print("\n[실제로 뽑힌 9개 조합]")
sampled = pd.DataFrame(random_search.cv_results_)[
    ["param_n_estimators", "param_max_depth",
     "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score").reset_index(drop=True)
sampled[["mean_test_score", "std_test_score"]] = \
    sampled[["mean_test_score", "std_test_score"]].round(4)
print(sampled.to_string(index=False))

탐색 공간 전체 조합 수 : 1,809개
실제 평가한 조합 수    : 9개 (n_iter)
탐색 비율              : 0.498%
총 학습 횟수           : 45회 (n_iter x cv 5)
탐색 시간              : 8.16초
best_params_           : {'max_depth': 11, 'n_estimators': 230}
best_score_            : 0.8210
mean_test_score        : 0.8210
std_test_score         : 0.0651

[실제로 뽑힌 9개 조합]
 param_n_estimators  param_max_depth  mean_test_score  std_test_score  rank_test_score
                230               11           0.8210          0.0651                1
                288               11           0.8209          0.0664                2
                216               11           0.8199          0.0659                3
                279               10           0.8186          0.0693                4
                202                8           0.8183          0.0640                5
                152                9           0.8179          0.0680                6
                203                7           0.8167          0.062

최고 CV평균 : 0.82
CV 표준편차 : 0.06
탐색시간 : 7.61

Grid search의 CV평균보다 약 0.0004 높다.  
반편에 표준편차는 0.006 정도로 커졌다.  
Grid search와 지금 모델을 비교하여 보면 평균 R^2은 높지만 표준편차(변동)가 조금 더 크다.


### Q2. Random Search가 Grid Search보다 유리할 수 있는 상황은 언제인가요?

**답변:**  
내가 어떤 값을 넣어야할지 모르고 모든 조합을 평가할 시간이 없을 때 주로 유용하다.

---

# 필수 2. Optuna와 CV 결합

## 배경

Grid와 Random은 이전 탐색 결과와 상관없이 후보를 선택합니다.

이번 실습에서는 Optuna의 `TPESampler`를 사용합니다.  
처음 3개의 trial은 초기 탐색으로 수행하고, 이후 trial부터는 앞선 결과를 참고하여 다음 후보를 제안하도록 설정합니다.

## 문제 1. Optuna objective에 교차검증을 연결하세요.

### 요구사항

1. `objective(trial)` 함수를 만듭니다.
2. 다음 하이퍼파라미터를 탐색합니다.
   - `n_estimators`: 100 ~ 300
   - `max_depth`: 4 ~ 12
3. `RandomForestRegressor`를 사용합니다.
4. `cross_val_score(..., cv=5, scoring="r2")`를 사용합니다.
5. CV 평균 점수를 반환합니다.
6. 다음 설정의 `TPESampler`를 만듭니다.

```python
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)
```

7. `direction="maximize"`와 위 sampler를 사용해 study를 만듭니다.
8. `n_trials=9`로 탐색합니다.
9. 탐색 시간을 측정합니다.
10. `best_params`, `best_value`를 출력합니다.

### 확인 포인트

- 한 번의 시도를 trial이라고 부르는가?
- 탐색 전체를 study라고 부르는가?
- 처음 3개 trial 이후에는 이전 trial 결과를 활용하는 TPE 탐색이 이루어지는가?
- CV 평균 성능을 Optuna의 최적화 대상으로 사용했는가?


In [11]:
# TODO
# Optuna objective + CV를 구성하세요.
# TPESampler(seed=42, n_startup_trials=3)를 사용하고
# 총 9 trial을 실행하세요.

# TODO
# Optuna 최적 조합의 CV 평균과 표준편차를 확인하세요.
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 300)
    max_depth = trial.suggest_int("max_depth", 4, 12)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
    return scores.mean()

sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)

study = optuna.create_study(direction="maximize", sampler=sampler)

start = time.time()
study.optimize(objective, n_trials=9)
optuna_time = time.time() - start

print(f"trial 수      : {len(study.trials)}개")
print(f"총 학습 횟수  : {len(study.trials) * 5}회 (trial x cv 5)")
print(f"탐색 시간     : {optuna_time:.2f}초")
print(f"best_params   : {study.best_params}")
print(f"best_value    : {study.best_value:.4f}")

print("\n[trial별 결과]")
trial_result = pd.DataFrame([{
    "trial": t.number,
    "단계": "초기 탐색" if t.number < 3 else "TPE 탐색",
    "n_estimators": t.params["n_estimators"],
    "max_depth": t.params["max_depth"],
    "cv_r2": round(t.value, 4)
} for t in study.trials])
trial_result["누적 최고"] = trial_result["cv_r2"].cummax()
print(trial_result.to_string(index=False))


trial 수      : 9개
총 학습 횟수  : 45회 (trial x cv 5)
탐색 시간     : 6.94초
best_params   : {'n_estimators': 175, 'max_depth': 12}
best_value    : 0.8213

[trial별 결과]
 trial     단계  n_estimators  max_depth  cv_r2  누적 최고
     0  초기 탐색           175         12 0.8213 0.8213
     1  초기 탐색           247          9 0.8185 0.8213
     2  초기 탐색           131          5 0.7990 0.8213
     3 TPE 탐색           109         12 0.8201 0.8213
     4 TPE 탐색           204         12 0.8206 0.8213
     5 TPE 탐색           274          4 0.7796 0.8213
     6 TPE 탐색           193          8 0.8180 0.8213
     7 TPE 탐색           109          7 0.8163 0.8213
     8 TPE 탐색           290         10 0.8187 0.8213


### Q3. Optuna가 Grid / Random과 다른 핵심은 무엇인가요?

**답변:**  
Grid는 정해둔 모든 후보 조합을 평가하고, Random은 정해진 횟수만큼 후보를 무작위로 선택  
Optuna는 초기 3개 trial이후 결과를 참고하여 상대적으로 유리한 영역을 더 깊게 탐색하는 방식을 사용  

## 문제 2. Optuna 최적 조합의 CV 표준편차를 확인하세요.

### 요구사항

1. `study.best_params`로 RandomForestRegressor를 만듭니다.
2. 다시 5-Fold `cross_val_score()`를 수행합니다.
3. CV 평균과 표준편차를 계산합니다.

### 확인 포인트

- 최종 후보를 볼 때 평균만 보지 않고 Fold 간 흔들림도 확인했는가?

In [13]:
# TODO
# Optuna 최적 조합의 CV 평균과 표준편차를 확인하세요.
import numpy as np

best_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)
best_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring="r2")

print(f"best_params : {study.best_params}")
print(f"Fold별 R2   : {np.round(best_scores, 4)}")
print(f"CV 평균     : {best_scores.mean():.4f}")
print(f"CV 표준편차 : {best_scores.std():.4f}")
print(f"평균 +- 표준편차 : {best_scores.mean() - best_scores.std():.4f} ~ "
      f"{best_scores.mean() + best_scores.std():.4f}")
print(f"최저 Fold / 최고 Fold : {best_scores.min():.4f} / {best_scores.max():.4f}")
print(f"\nstudy.best_value 와 일치 여부 : "
      f"{np.isclose(best_scores.mean(), study.best_value)}")


best_params : {'n_estimators': 175, 'max_depth': 12}
Fold별 R2   : [0.8694 0.6967 0.828  0.8428 0.8694]
CV 평균     : 0.8213
CV 표준편차 : 0.0643
평균 +- 표준편차 : 0.7570 ~ 0.8855
최저 Fold / 최고 Fold : 0.6967 / 0.8694

study.best_value 와 일치 여부 : True


앞서 진행한 세가지 모델 결과 비교  
grid의 표준편차는 약 0.065 / Random은 0.065 / optuna는 0.0642이다.  
-> optuna를 통해 결과를 구했을 때 평균이 가장 높고, 표준편차가 가장 적었다.  

---

# 과제 1. 최종 모델 선정과 hold-out Test 확인

## 배경

세 탐색 전략에서 각각 최적 후보를 얻었습니다.

최종 모델은 **CV 평균을 우선 기준으로 비교**하되, 성능 차이가 매우 작다면 다음 항목도 함께 확인합니다.

- CV 표준편차
- 탐색 시간

그리고 최종 후보를 정한 뒤 **hold-out Test는 딱 한 번만** 사용합니다.

> 탐색 시간은 각 탐색 도구의 내부 동작과 실행 환경에 따라 포함되는 작업이 조금 다를 수 있으므로, 이번 실습에서는 절대적인 속도 순위가 아니라 **대략적인 탐색 비용 비교**로 해석합니다.

## 요구사항

1. Grid / Random / Optuna 결과를 다음 컬럼으로 하나의 DataFrame에 정리합니다.
   - `search_method`
   - `best_params`
   - `cv_mean`
   - `cv_std`
   - `search_seconds`
2. 세 후보를 비교합니다.
3. CV 평균을 우선 기준으로 최종 후보를 선택하고, 평균이 거의 비슷하다면 CV 표준편차와 탐색 비용도 함께 고려하여 선택 이유를 설명합니다.
4. 튜닝하지 않은 기준 모델(`RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)`)과 3번에서 선택한 튜닝 후보를 비교하세요. 필수 실습의 `cv=5`와 동일한 `KFold(n_splits=5, shuffle=False)` 분할을 두 모델에 공통 적용하고, `cross_validate(..., scoring="r2", return_train_score=True)`로 평가하세요.
   - Fold별 Train R²와 검증 R²를 `fold_comparison`으로 정리합니다.
   - Train R² 평균, 검증 R² 평균·표준편차, `Train 평균 - 검증 평균`을 `tuning_comparison`으로 정리합니다.
   - `튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²`를 계산합니다. R² 차이값으로 보고하며, 백분율 개선율로 바꾸지 않습니다.
   - 평균·변동·Train–검증 차이로 개선 여부를 설명합니다. 튜닝이 개선되지 않았다면 기준 모델 유지도 가능합니다. Test 확인 전에 최종 모델과 선택 이유를 확정합니다.
   - 최종 선택 모델을 Train 전체에 다시 학습합니다.
5. hold-out Test에서 `.score()`로 최종 R²를 딱 한 번 확인합니다.
6. 최종 선택 모델의 CV 평균과 hold-out Test R²의 차이를 계산하고, 과적합 가능성과 일반화 판단의 한계를 설명합니다. Test 결과를 본 뒤 다시 튜닝하거나 후보를 교체하지 않습니다.
7. 선택한 모델을 `model_artifacts/best_model.pkl`로 저장합니다.
8. 다음 정보를 `model_artifacts/model_meta.json`에 저장합니다.
   - 모델 이름
   - 저장 시각
   - 선택한 탐색 방법
   - 최적 파라미터
   - CV 평균
   - CV 표준편차
   - 최종 Test R²
   - 기준 모델 설정, 튜닝 후보 설정·탐색 방법
   - 튜닝 전후 CV 비교 결과와 CV 평균 R² 차이
   - 선택 모델 CV 평균과 Test R²의 차이

### 확인 포인트

- Test 데이터를 탐색 중간에 사용하지 않았는가?
- CV 평균을 우선으로 보고, 필요할 때 표준편차와 탐색 시간도 함께 고려했는가?
- 최종 모델을 정한 뒤 Test를 한 번만 확인했는가?
- 모델뿐 아니라 메타데이터도 저장했는가?

- 튜닝 전후를 동일한 Fold에서 비교하고 개선 여부를 수치로 설명했는가?
- CV의 `test_score`는 Fold 내부 검증 점수이며 hold-out Test 점수가 아님을 구분했는가?
- 작은 표준편차나 CV/Test 점수의 일치만으로 일반화가 보장된다고 단정하지 않았는가?


In [14]:
# TODO
# 세 탐색 후보를 비교하고 기준 모델과 튜닝 후보의 동일 Fold CV 결과를 정리하세요.
# 튜닝 전후 R² 차이를 계산해 최종 선택을 확정한 뒤
# 선택 모델만 hold-out Test에서 한 번 평가하고 모델·비교 메타데이터를 저장하세요.
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score,
    KFold,              # 추가
    cross_validate      # 추가
)

# 요구사항 1~3
grid_idx, random_idx = grid_search.best_index_, random_search.best_index_

search_comparison = pd.DataFrame([
    {"search_method": "GridSearchCV",
     "best_params": grid_search.best_params_,
     "cv_mean": round(grid_search.cv_results_["mean_test_score"][grid_idx], 4),
     "cv_std": round(grid_search.cv_results_["std_test_score"][grid_idx], 4),
     "search_seconds": round(grid_time, 2)},
    {"search_method": "RandomizedSearchCV",
     "best_params": random_search.best_params_,
     "cv_mean": round(random_search.cv_results_["mean_test_score"][random_idx], 4),
     "cv_std": round(random_search.cv_results_["std_test_score"][random_idx], 4),
     "search_seconds": round(random_time, 2)},
    {"search_method": "Optuna (TPE)",
     "best_params": study.best_params,
     "cv_mean": round(best_scores.mean(), 4),
     "cv_std": round(best_scores.std(), 4),
     "search_seconds": round(optuna_time, 2)},
])

print("[search_comparison]")
print(search_comparison.to_string(index=False))

best_row = search_comparison.loc[search_comparison["cv_mean"].idxmax()]
print(f"\ncv_mean 최대-최소 차이 : "
      f"{search_comparison['cv_mean'].max() - search_comparison['cv_mean'].min():.4f}")
print(f"cv_std 평균            : {search_comparison['cv_std'].mean():.4f}")
print(f"\n선택한 탐색 방법   : {best_row['search_method']}")
print(f"선택한 파라미터    : {best_row['best_params']}")
print(f"CV 평균 / 표준편차 : {best_row['cv_mean']:.4f} / {best_row['cv_std']:.4f}")
print(f"탐색 시간          : {best_row['search_seconds']}초")

selected_method = best_row["search_method"]
selected_params = best_row["best_params"]


# 요구사항 4
cv_kfold = KFold(n_splits=5, shuffle=False)

baseline_model = RandomForestRegressor(
    n_estimators=100, max_depth=None, random_state=42, n_jobs=-1
)
tuned_model = RandomForestRegressor(
    **selected_params, random_state=42, n_jobs=-1
)

cv_results = {}
for name, model in [("기준 모델", baseline_model), ("튜닝 후보", tuned_model)]:
    cv_results[name] = cross_validate(
        model, X_train, y_train, cv=cv_kfold,
        scoring="r2", return_train_score=True
    )

fold_comparison = pd.DataFrame({
    "fold": range(1, 6),
    "기준_train_r2": cv_results["기준 모델"]["train_score"].round(4),
    "기준_val_r2":   cv_results["기준 모델"]["test_score"].round(4),
    "튜닝_train_r2": cv_results["튜닝 후보"]["train_score"].round(4),
    "튜닝_val_r2":   cv_results["튜닝 후보"]["test_score"].round(4),
})
fold_comparison["val_차이"] = (
    fold_comparison["튜닝_val_r2"] - fold_comparison["기준_val_r2"]).round(4)

print("[fold_comparison]")
print(fold_comparison.to_string(index=False))

rows = []
for name in ["기준 모델", "튜닝 후보"]:
    tr = cv_results[name]["train_score"]
    va = cv_results[name]["test_score"]
    rows.append({
        "model": name,
        "train_r2_mean": round(tr.mean(), 4),
        "val_r2_mean": round(va.mean(), 4),
        "val_r2_std": round(va.std(), 4),
        "train_minus_val": round(tr.mean() - va.mean(), 4),
    })
tuning_comparison = pd.DataFrame(rows)

print("\n[tuning_comparison]")
print(tuning_comparison.to_string(index=False))

base_cv = tuning_comparison.loc[0, "val_r2_mean"]
tuned_cv = tuning_comparison.loc[1, "val_r2_mean"]
cv_diff = round(tuned_cv - base_cv, 4)

print(f"\n기준 모델 CV 평균 R2 : {base_cv:.4f}")
print(f"튜닝 후보 CV 평균 R2 : {tuned_cv:.4f}")
print(f"CV 평균 R2 차이 (튜닝 - 기준) : {cv_diff:+.4f}")
print(f"튜닝 후보가 더 높은 Fold 수   : "
      f"{int((fold_comparison['val_차이'] > 0).sum())} / 5")
print(f"\n판정 : {'튜닝 후보 채택' if cv_diff > 0 else '기준 모델 유지'}")

final_model = tuned_model if cv_diff > 0 else baseline_model
final_label = "튜닝 후보" if cv_diff > 0 else "기준 모델"
final_cv_mean = tuned_cv if cv_diff > 0 else base_cv
final_cv_std = (tuning_comparison.loc[1, "val_r2_std"] if cv_diff > 0
                else tuning_comparison.loc[0, "val_r2_std"])

final_model.fit(X_train, y_train)
print("\nTrain 전체 재학습 완료")

# 요구사항 5

test_r2 = final_model.score(X_test, y_test)

print(f"최종 모델 CV 평균 R2 : {final_cv_mean:.4f}")
print(f"hold-out Test R2     : {test_r2:.4f}")
print(f"차이 (Test - CV)     : {test_r2 - final_cv_mean:+.4f}")

os.makedirs("model_artifacts", exist_ok=True)
joblib.dump(final_model, "model_artifacts/best_model.pkl")

model_meta = {
    "model_name": type(final_model).__name__,
    "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "selected_search_method": selected_method,
    "best_params": selected_params,
    "cv_mean": round(float(final_cv_mean), 4),
    "cv_std": round(float(final_cv_std), 4),
    "test_r2": round(float(test_r2), 4),

    "baseline_config": {
        "n_estimators": 100, "max_depth": None,
        "random_state": 42, "n_jobs": -1
    },
    "tuned_config": {**selected_params, "random_state": 42, "n_jobs": -1},
    "tuned_search_method": selected_method,

    "tuning_comparison": tuning_comparison.to_dict(orient="records"),
    "cv_mean_r2_diff": float(cv_diff),
    "final_model_label": final_label,
    "cv_mean_minus_test_r2": round(float(final_cv_mean - test_r2), 4),
}

with open("model_artifacts/model_meta.json", "w", encoding="utf-8") as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=2)

print("\n저장된 파일 :")
for fn in sorted(os.listdir("model_artifacts")):
    size = os.path.getsize(f"model_artifacts/{fn}")
    print(f"  model_artifacts/{fn}  ({size:,} bytes)")

print("\n[model_meta.json 내용]")
print(json.dumps(model_meta, ensure_ascii=False, indent=2))

[search_comparison]
     search_method                            best_params  cv_mean  cv_std  search_seconds
      GridSearchCV {'max_depth': 12, 'n_estimators': 200}   0.8206  0.0644            7.21
RandomizedSearchCV {'max_depth': 11, 'n_estimators': 230}   0.8210  0.0651            8.16
      Optuna (TPE) {'n_estimators': 175, 'max_depth': 12}   0.8213  0.0643            6.94

cv_mean 최대-최소 차이 : 0.0007
cv_std 평균            : 0.0646

선택한 탐색 방법   : Optuna (TPE)
선택한 파라미터    : {'n_estimators': 175, 'max_depth': 12}
CV 평균 / 표준편차 : 0.8213 / 0.0643
탐색 시간          : 6.94초
[fold_comparison]
 fold  기준_train_r2  기준_val_r2  튜닝_train_r2  튜닝_val_r2  val_차이
    1       0.9717     0.8627       0.9715     0.8694  0.0067
    2       0.9754     0.6808       0.9764     0.6967  0.0159
    3       0.9750     0.8343       0.9721     0.8280 -0.0063
    4       0.9746     0.8450       0.9726     0.8428 -0.0022
    5       0.9717     0.8723       0.9709     0.8694 -0.0029

[tuning_comparison]
model  train_

### Q4. CV 평균이 아주 조금 높은 모델의 탐색 시간이 몇 배 더 길다면 무조건 그 모델을 선택해야 하나요?

**답변:**  
- 아니오  

-> CV 평균 차이가 Fold 간 변동보다 작으면 그 차이가 실제 성능 때문인지 분할 운 때문인지 구분할 수 없음.  
이번 실습의 세 후보 cv_mean은 0.8206 / 0.8210 / 0.8213으로 차이가 0.0007인데 cv_std 평균은 0.0646임.  
-> CV 평균으로만 판단하지 않고 CV 표준편차와 탐색 비용을 함께 고려하여 선택해야 함.

### Q5. hold-out Test를 탐색 마지막에 한 번만 사용해야 하는 이유는 무엇인가요?

**답변:**   
- Test 점수를 보고 모델이나 파라미터를 고치면 그 선택에 Test 정보가 반영되므로, 처음 보는 데이터의 성능보다 높게 나올 수 있기 때문임  
-> 따라서 탐색과 선택은 Train 안에서만 수행하고, 최종 모델이 정해진 뒤 Test를 한 번만 사용해야 함.

### Q6. 튜닝 전후 CV 평균 R²는 얼마나 달라졌나요? Fold별 변동과 Train–검증 점수 차이를 근거로 개선 여부를 설명하세요.

**답변:** 
- CV 평균 R² 0.8190 -> 0.8213, `+0.0023 증가함`  

    - Fold별 변동 : 검증 R² 표준편차가 0.0704 -> 0.0643으로 0.0061 감소  
-> 5개 Fold 중 2개만 증가, 3개는 감소(Fold1 +0.0067 / Fold2 +0.0159 / Fold3 −0.0063 / Fold4 −0.0022 / Fold5 −0.0029)  
-> 그중 Fold2는 +0.0159로 5개 Fold중 제일 변동이 크고, 기준 모델과 튜닝 모델 모두 Fold2의 검증 R²(0.6808 / 0.6967)가 나머지 4개 Fold보다 낮게 나와 비교적 설명력이 낮은 것으로 해석할 수 있음.  

    - Train–검증 점수 차이 : 0.1547 -> 0.1514로 0.0033 감소  
-> 다만 기준 모델과 튜닝 모델 모두 Train R² 0.97정도인데, 검증 R² 0.82정도로 격차 0.15 정도가 있어 과적합 경향이 해소되었다고 보기는 어려움  

-> `평균, 변동, Train–검증 차이 모두 개선했으므로 튜닝 모델을 채택하되, 개선 폭이 CV 표준편차(0.0643)보다 작고 감소된 Fold가 3개이므로 뚜렷한 개선으로 보기는 어려움.`

### Q7. CV 결과와 최종 Test 결과를 어떻게 해석했나요? 과적합 가능성과 일반화 판단의 한계를 설명하세요.

**답변:**

- CV 평균 R² 0.8213 / hold-out Test R² 0.8911, `차이 +0.0698로 Test가 더 높게 나옴`  

    - CV의 test_score는 Train을 5등분하여 그중 1/5로 평가한 Fold 내부 검증 점수 (hold-out Test 점수가 아님)  
-> 각 Fold는 Train의 4/5인 934행으로 학습했으나 최종 모델은 Train 전체 1,168행으로 재학습했으므로, 학습 데이터 양이 다름  
-> 또한 CV 평균이 Fold2(0.6967)에 의해 낮아져 있으며, Fold2를 제외한 CV 평균은 0.8524로 Test 0.8911과의 차이가 +0.0387로 줄어듦  

    - `과적합 가능성` : Test R²가 높게 나왔다는 사실만으로 과적합이 없다고 볼 수 없음  
-> Train R² 평균 0.9727, 검증 R² 평균 0.8213의 격차 0.1514 정도 있으며 학습 곡선으로 두 곡선의 격차가 좁혀지는지 평평하게 유지되는지 확인해야 함  

    - `일반화 판단의 한계`  
-> Test는 292건 한 번의 측정이므로 분할이 달라지면 점수도 달라질 수 있음  
-> CV 표준편차가 작다는 것이나 CV와 Test 점수가 일치한다는 것만으로 일반화가 보장되지는 않음
-> 재튜닝 없이 1회로 종료했으므로 Test 점수는 이 모델의 성능 추정치이며 확정값이 아님  


---

# 실습 마무리

1. GridSearchCV와 RandomizedSearchCV의 가장 큰 탐색 방식 차이는 무엇인가요?  
- Grid : 주어진 모든 후보 조합 평가  
- Random : n_iter 만큼 무작위 조합 평가  

2. RandomizedSearchCV의 `n_iter`는 무엇을 의미하나요?  
- Random Search가 평가할 조합의 개수, 탐색 예산을 의미  

3. Optuna에서 `study`와 `trial`은 각각 무엇인가요?  
- study : 전체 탐색  
- trial : 한 번의 시도  

4. 최종 모델을 고를 때 CV 평균 외에 무엇을 함께 봐야 하나요?  
-> CV 평균, CV 표준편차, 탐색 학습 비용  

5. hold-out Test는 언제 사용해야 하나요?  
-> 모든 탐색과 모델 선택이 끝난 뒤 마지막에 한 번 사용한다.  

6. 최종 튜닝 모델과 메타데이터를 저장하는 이유는 무엇인가요?  
-> 나중에 같은 모델을 재사용하고, 어떤 조건으로 만들어졌는지 추적(Log)하여 재현성과 운영 관리를 높이기 위해 저장한다.